# G1_01 — HTTP con le mani

> **Corso ITS D.E.Mo.S. — Laboratorio API + Frontend — Giornata 1**
> Esegui le celle dall'alto verso il basso con **Shift+Invio**. Prima di ogni cella c'è scritto **cosa aspettarti**.
> Se qualcosa non torna, alza la mano: l'errore si legge insieme.

## Cosa faremo (45 minuti)

Oggi pomeriggio costruirai un'API. Ma prima devi **vederne una dal lato di chi la usa**.

Un'API HTTP è un programma che sta su un server e risponde a richieste. La richiesta la fai tu (o il tuo codice, o il browser).
Ogni richiesta ha:

| Pezzo | Cos'è | Esempio |
|-------|-------|---------|
| **Metodo** | Il verbo: cosa voglio fare | `GET` (leggi), `POST` (crea), `PUT` (modifica), `DELETE` (cancella) |
| **URL** | Dove | `https://.../tickets` |
| **Header** | Informazioni di contorno | `Content-Type: application/json`, `X-API-Key: ...` |
| **Body** | I dati che mando (solo per POST/PUT) | `{"title": "Stampante rotta"}` |

E ogni risposta ha uno **status code** (un numero) e un **body** (di solito JSON).

Il bersaglio di oggi è l'API di gestione ticket del docente, già online. È la stessa che costruirai tu nel pomeriggio.

### Cella 1 — Da dove partiamo

Impostiamo l'indirizzo dell'API. Se l'API del docente non risponde (può capitare: è un servizio gratuito che "si addormenta"),
la cella passa da sola a un'API pubblica di riserva.

**Cosa aspettarti:** una riga che dice quale API useremo.

In [ ]:
import requests

API_URL = "https://portale-ticket-its-mdrp.onrender.com"   # l'API del docente (scritta alla lavagna se cambia)

try:
    # timeout=30: il primo risveglio dell'API puo' richiedere 20-30 secondi ("cold start")
    risposta = requests.get(API_URL + "/health", timeout=30)
    risposta.raise_for_status()                      # 4xx o 5xx -> errore
    viva = risposta.json().get("status") == "ok"     # deve essere JSON, non una pagina web
except (requests.exceptions.RequestException, ValueError):
    viva = False

# Perche' non basta guardare lo status code: una pagina di login o di errore
# risponde lo stesso "200", ma con dentro HTML. Se non riusciamo a leggere il
# JSON, quello che c'e' dall'altra parte non e' la nostra API.
if not viva:
    print("API del docente non raggiungibile. Uso l'API di riserva.")
    API_URL = "https://jsonplaceholder.typicode.com"   # riserva: risorsa /todos invece di /tickets
else:
    print("API del docente raggiungibile.")

RESOURCE = "/todos" if "typicode" in API_URL else "/tickets"
print("Useremo:", API_URL + RESOURCE)

### Cella 2 — La prima GET

`GET` vuol dire "dammi". Chiediamo la lista dei ticket.

**Cosa aspettarti:** `200` e un pezzo di testo che inizia con `[{`. Quello è JSON: una lista di oggetti.

In [ ]:
response = requests.get(API_URL + RESOURCE, timeout=30)

print("Status code:", response.status_code)
print("Primi 300 caratteri del body:")
print(response.text[:300])

### Cella 3 — Lo status code è un semaforo

Il numero che torna dice **com'è andata**, prima ancora di leggere il body.

| Famiglia | Significato | Quelli che vedrai oggi |
|----------|-------------|------------------------|
| **2xx** | Tutto bene | `200 OK`, `201 Created`, `204 No Content` |
| **4xx** | Hai sbagliato tu (client) | `401` chiave mancante, `404` non esiste, `422` dati sbagliati |
| **5xx** | Ha sbagliato il server | `500` bug nel codice del server |

Proviamo a chiedere qualcosa che non esiste.

**Cosa aspettarti:** `404`.

In [ ]:
response = requests.get(API_URL + RESOURCE + "/999999", timeout=30)
print("Status code:", response.status_code)
print("Body:", response.text)

### Cella 4 — Dal testo al dizionario Python

Il body è **testo** in formato JSON. Per usarlo in Python lo trasformiamo con `.json()`:
diventa una **lista di dizionari**. Poi possiamo fare le cose normali: `len()`, ciclo `for`, accesso con le chiavi.

**Cosa aspettarti:** il numero di elementi e il titolo di ognuno, uno per riga.

In [ ]:
response = requests.get(API_URL + RESOURCE, timeout=30)
items = response.json()          # da testo JSON a lista Python

print("Tipo:", type(items))
print("Quanti elementi:", len(items))
print()

for item in items[:10]:          # i primi 10, per non riempire lo schermo
    print(f"#{item['id']:>3}  {item['title']}")

### Cella 5 — Guardiamo un solo elemento

Un elemento della lista è un **dizionario**: coppie chiave → valore. Le chiavi sono i "campi" del ticket.

**Cosa aspettarti:** le chiavi del primo elemento e poi il dizionario intero.

In [ ]:
first = items[0]

print("Chiavi disponibili:", list(first.keys()))
print()
print("Il primo elemento completo:")
print(first)

### Cella 6 — Gli header: le informazioni di contorno

Ogni risposta porta degli **header**: non sono i dati, sono informazioni *sui* dati.
Il più importante è `Content-Type`: dice al client come interpretare il body.

**Cosa aspettarti:** `application/json` tra gli header.

In [ ]:
for name, value in response.headers.items():
    print(f"{name}: {value}")

print()
print("Content-Type:", response.headers.get("Content-Type"))

### Cella 7 — Filtrare con i query parameter

Dopo il `?` nell'URL si possono passare **parametri**: `?status=aperto`. Il server li legge e restringe la risposta.

**Cosa aspettarti:** solo i ticket con stato `aperto` (con l'API di riserva: solo i `todos` completati).

In [ ]:
if RESOURCE == "/tickets":
    params = {"status": "aperto"}
else:
    params = {"completed": "true"}

response = requests.get(API_URL + RESOURCE, params=params, timeout=30)

print("URL chiamato davvero:", response.url)   # guarda cosa e' finito dopo il '?'
print("Status:", response.status_code, "- elementi:", len(response.json()))

### Cella 8 — Provare a scrivere: POST

Per **creare** un ticket si usa `POST` con un body JSON. `requests` lo manda con `json=...`.

Ma l'API del docente protegge le scritture: senza la chiave `X-API-Key` risponde **401 Unauthorized**.
È voluto: pomeriggio lo farai anche tu.

**Cosa aspettarti:** `401` con l'API del docente (l'API di riserva invece fa finta di accettare e risponde `201`).

In [ ]:
new_ticket = {"title": "Prova dal notebook", "description": "Creato da G1_01"}

response = requests.post(API_URL + RESOURCE, json=new_ticket, timeout=30)

print("Status code:", response.status_code)
print("Body:", response.text)

### Cella 9 — POST con la chiave

Adesso aggiungiamo l'header con la chiave. Il docente la scrive alla lavagna: **incollala nella variabile qui sotto**.

**Cosa aspettarti:** `201 Created` e il ticket restituito **con un `id` e una data** decisi dal server.
Riesegui la Cella 4: il tuo ticket è in lista. Chi l'ha creato è scritto nel titolo, quindi... occhio a cosa scrivi.

In [ ]:
API_KEY = "INCOLLA-QUI-LA-CHIAVE-DEL-DOCENTE"

headers = {"X-API-Key": API_KEY}
new_ticket = {"title": "Prova dal notebook", "description": "Creato da G1_01"}

response = requests.post(API_URL + RESOURCE, json=new_ticket, headers=headers, timeout=30)

print("Status code:", response.status_code)
print("Body:", response.text)

### Cella 10 — Mandare dati sbagliati

Cosa succede se il titolo è troppo corto? L'API **controlla** e risponde `422 Unprocessable Entity`,
spiegando *quale* campo è sbagliato e *perché*. Questo è il tema del prossimo notebook.

**Cosa aspettarti:** `422` e un messaggio che parla di `title` e di "at least 3 characters" (con l'API di riserva: `201`, perché non controlla nulla — ed è un problema).

In [ ]:
bad_ticket = {"title": "ab"}

response = requests.post(API_URL + RESOURCE, json=bad_ticket, headers=headers, timeout=30)

print("Status code:", response.status_code)
print("Body:", response.text)

## Riepilogo

- Una richiesta HTTP = **metodo + URL + header + body**. Una risposta = **status code + header + body**.
- `GET` legge, `POST` crea, `PUT` modifica, `DELETE` cancella.
- Lo status code si legge **prima** del body: 2xx bene, 4xx colpa mia, 5xx colpa del server.
- Il body JSON diventa lista/dizionario Python con `.json()`.
- Le scritture vanno protette: senza chiave → `401`. I dati sbagliati vanno rifiutati: → `422`.

## Mini-esercizio (se hai finito prima)

1. Fai una `GET` di un singolo ticket usando l'`id` di quello che hai creato: `API_URL + RESOURCE + "/" + str(id)`.
2. Stampa solo `status code` e `title`.